# 38. SegFormer 전체 Forward 흐름

이 노트북은 `37_SegFormer_MLP_Decoder_구조.ipynb` 다음 단계로, SegFormer의 encoder와 decoder를 하나의 forward pipeline으로 연결합니다.

실제 SegFormer 구현은 PyTorch module로 구성되지만, 여기서는 shape 중심의 NumPy 예제로 전체 흐름을 따라갑니다.

이번 노트북의 목표는 다음과 같습니다.

- 입력 이미지에서 stage feature가 만들어지는 흐름을 정리합니다.
- encoder output이 decoder input으로 들어가는 shape를 확인합니다.
- segmentation logits와 최종 mask prediction의 관계를 이해합니다.
- 실제 코드 구현 시 shape mismatch를 점검하는 기준을 세웁니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(7)

## 38-1. Toy SegFormer pipeline

아래 함수는 실제 학습 모델이 아니라 shape 흐름을 보여 주는 단순화된 forward입니다.

```text
image
  -> encoder stage features
  -> MLP projection
  -> upsample to H/4
  -> concat
  -> class logits
  -> upsample to input size
```

In [ ]:
def make_stage_features(image, channels=(16, 32, 64, 128)):
    h, w, _ = image.shape
    features = []
    for stride, c in zip([4, 8, 16, 32], channels):
        fh, fw = h // stride, w // stride
        base = image.reshape(fh, stride, fw, stride, 3).mean(axis=(1, 3))
        weight = np.random.normal(scale=0.2, size=(3, c))
        feat = base.reshape(-1, 3) @ weight
        features.append(feat.reshape(fh, fw, c))
    return features

def upsample_nearest(feature, target_h, target_w):
    h, w, c = feature.shape
    return np.repeat(np.repeat(feature, target_h // h, axis=0), target_w // w, axis=1)

def toy_segformer_forward(image, num_classes=4, decoder_dim=24):
    features = make_stage_features(image)
    target_h, target_w = features[0].shape[:2]
    decoded = []
    for feat in features:
        h, w, c = feat.shape
        proj = feat.reshape(-1, c) @ np.random.normal(scale=0.1, size=(c, decoder_dim))
        proj = proj.reshape(h, w, decoder_dim)
        decoded.append(upsample_nearest(proj, target_h, target_w))
    fused = np.concatenate(decoded, axis=-1)
    logits = fused.reshape(-1, fused.shape[-1]) @ np.random.normal(scale=0.05, size=(fused.shape[-1], num_classes))
    logits = logits.reshape(target_h, target_w, num_classes)
    full_logits = upsample_nearest(logits, image.shape[0], image.shape[1])
    return features, logits, full_logits, full_logits.argmax(axis=-1)

In [ ]:
image = np.zeros((224, 224, 3), dtype=np.float32)
image[:, :, :] = [0.75, 0.80, 0.85]
image[45:145, 40:120, :] = [0.9, 0.35, 0.25]
image[90:185, 135:200, :] = [0.25, 0.55, 0.95]

features, low_logits, full_logits, pred = toy_segformer_forward(image)
print('input:', image.shape)
for i, f in enumerate(features, start=1):
    print(f'encoder C{i}: {f.shape}')
print('low-resolution logits:', low_logits.shape)
print('full-resolution logits:', full_logits.shape)
print('prediction:', pred.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(image)
axes[0].set_title('input image')
axes[1].imshow(low_logits.argmax(axis=-1), cmap='tab10')
axes[1].set_title('H/4 logits argmax')
axes[2].imshow(pred, cmap='tab10')
axes[2].set_title('upsampled prediction')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 38-2. 개발 중 shape 체크 포인트

SegFormer 계열 구현에서 자주 확인해야 하는 지점은 다음과 같습니다.

- Encoder stage output이 `B, C, H, W`인지 `B, H, W, C`인지 명확히 확인합니다.
- Decoder projection 전후 channel dimension이 의도한 값인지 확인합니다.
- 모든 stage feature가 같은 spatial size로 upsample되었는지 확인합니다.
- 최종 logits shape가 `B, num_classes, H, W`인지 확인합니다.
- label mask shape와 loss input shape가 맞는지 확인합니다.

## 정리

- SegFormer forward는 encoder multi-scale feature와 decoder fusion으로 볼 수 있습니다.
- Decoder는 stage feature를 projection, upsample, concat한 뒤 logits를 만듭니다.
- 최종 prediction은 logits의 class dimension에 대해 argmax를 취해 얻습니다.
- 다음 노트북 `39_SegFormer_간단_실습.ipynb`에서는 작은 synthetic segmentation 예제로 입력, label, prediction, IoU 계산을 실습합니다.